# Objective:
In this notebook, I transform raw scraped data into clean, ML-ready format with proper feature engineering for Random Forest model.

In [ ]:
# @title Block 15: Setup & Load Raw Data

# Mount Google Drive when running in Colab (no-op locally)
IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

# Import libraries
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("BLOCK 15: SETUP & LOAD RAW DATA")

# Environment-aware BASE_DIR with GUI folder selection for local environments
if 'google.colab' in sys.modules or IN_COLAB:
    # Colab: use Google Drive path
    BASE_DIR = Path('/content/drive') / 'My Drive' / 'Course' / 'Minor in AI' / 'Final Project' / 'ISRO Launch Trend Analysis - Apurva Upadhyay'
    print("Running in Colab: using Google Drive mount path")
else:
    # Local: offer GUI folder selection via tkinter
    try:
        import tkinter as tk
        from tkinter import filedialog
        
        print("Running locally: opening folder selection dialog...")
        print("Please select your project folder (should contain 'Dataset' subfolder with raw data)")
        
        # Create hidden root window for dialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        
        # Show folder picker dialog
        selected_path = filedialog.askdirectory(
            title="Select Project Folder (contains Dataset/isro_launch_history_raw.csv)",
            initialdir=os.path.expanduser("~")
        )
        
        root.destroy()
        
        if selected_path:
            BASE_DIR = Path(selected_path)
            print(f"Selected project folder: {BASE_DIR}")
        else:
            # Fallback: use current working directory
            print("No folder selected; using current working directory as fallback")
            BASE_DIR = Path.cwd()
            print(f"Fallback BASE_DIR: {BASE_DIR}")
    
    except ImportError:
        # Fallback if tkinter unavailable (rare on Windows/Mac, possible on headless Linux)
        print("WARNING: tkinter not available; using current working directory")
        print("Expected folder structure:")
        print("  <project_folder>/")
        print("    ├── Dataset/")
        print("    │   ├── isro_launch_history_raw.csv")
        print("    │   └── processed/  (will be created)")
        print("    └── 02_Data_Preprocessing.ipynb")
        BASE_DIR = Path.cwd()

RAW_DATA_PATH = BASE_DIR / 'Dataset' / 'isro_launch_history_raw.csv'

# Load data
try:
    df_raw = pd.read_csv(RAW_DATA_PATH)
    print(f"\n Successfully loaded raw data")
    print(f"   Path: {RAW_DATA_PATH}")
    print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
    print(f"   Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024:.2f} KB")

    # Display first few rows
    print(f"\n First 3 records:")
    print(df_raw.head(3).to_string())

except FileNotFoundError:
    print(f"ERROR: Raw data file not found at {RAW_DATA_PATH}")
    print("\n Please run 01_Data_Acquisition.ipynb first or update BASE_DIR to the correct path.")
    print("\n Expected folder structure:")
    print(f"  {BASE_DIR}/")
    print(f"    ├── Dataset/")
    print(f"    │   └── isro_launch_history_raw.csv")
    print(f"    └── 02_Data_Preprocessing.ipynb")
    raise FileNotFoundError(f"Raw data file not found at {RAW_DATA_PATH}")
except Exception as e:
    print(f"ERROR: {e}")
    raise


In [ ]:
# @title Block 16: Initial Data Inspection

print("BLOCK 16: INITIAL DATA INSPECTION")

# Data info
print("\n COLUMN INFORMATION:")
print(df_raw.dtypes)

# Missing values
print("\n MISSING VALUES:")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print("No missing values found! ")

# Duplicates
duplicates = df_raw.duplicated().sum()
print(f"\n DUPLICATE ROWS: {duplicates}")

# Unique values per column
print("\n UNIQUE VALUE COUNTS:")
for col in df_raw.columns:
    n_unique = df_raw[col].nunique()
    print(f"   {col:30s}: {n_unique:4d} unique values")

# Value distributions for key categorical columns
print("\n KEY CATEGORICAL DISTRIBUTIONS:")
print("\n Rocket Type:")
print(df_raw['rocket_type'].value_counts().to_string())

print("\n Launch Outcome:")
print(df_raw['launch_outcome'].value_counts().to_string())



In [ ]:
# @title Block 17: Date/Time Processing

from datetime import datetime
import re


print("BLOCK 17: DATE/TIME PROCESSING")


# Create working copy
df = df_raw.copy()

# Parse date_time_utc
def parse_datetime(date_str):
    """Parse various datetime formats from Wikipedia."""
    if pd.isna(date_str):
        return None

    date_str = str(date_str).strip()

    # Try different formats
    formats = [
        '%d %B %Y%H:%M',        # "20 September 199305:12"
        '%d %B %Y %H:%M',       # "20 September 1993 05:12"
        '%d %b %Y%H:%M',        # "20 Sep 199305:12"
        '%d %b %Y %H:%M',       # "20 Sep 1993 05:12"
        '%Y-%m-%d %H:%M:%S',    # "1993-09-20 05:12:00"
        '%Y-%m-%d',             # "1993-09-20"
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue

    # Last resort: let pandas infer
    try:
        return pd.to_datetime(date_str)
    except:
        return None

print("\n Parsing datetime column...")
df['launch_datetime'] = df['date_time_utc'].apply(parse_datetime)

# Check parsing success
parsed = df['launch_datetime'].notna().sum()
total = len(df)
print(f"Successfully parsed: {parsed}/{total} ({parsed/total*100:.1f}%)")

if parsed < total:
    failed = df[df['launch_datetime'].isna()]['date_time_utc'].head()
    print(f"\n   Failed to parse {total-parsed} dates. Examples:")
    print(failed.to_string(index=False))

# Extract temporal features
print("\n Extracting temporal features...")
df['launch_year'] = df['launch_datetime'].dt.year
df['launch_month'] = df['launch_datetime'].dt.month
df['launch_day_of_week'] = df['launch_datetime'].dt.dayofweek  # Monday=0
df['launch_hour'] = df['launch_datetime'].dt.hour
df['launch_quarter'] = df['launch_datetime'].dt.quarter

# Calculate days since previous launch (by rocket type)
print("   Computing days_since_previous_launch...")
df = df.sort_values(['rocket_type', 'launch_datetime']).reset_index(drop=True)
df['days_since_previous_launch'] = df.groupby('rocket_type')['launch_datetime'].diff().dt.days

# Fill first launch of each type with 0
df['days_since_previous_launch'] = df['days_since_previous_launch'].fillna(0)

# Display results
print(f"\n Temporal features created:")
print(f"   - launch_year ({df['launch_year'].min():.0f} to {df['launch_year'].max():.0f})")
print(f"   - launch_month (1-12)")
print(f"   - launch_day_of_week (0=Monday, 6=Sunday)")
print(f"   - launch_hour (0-23)")
print(f"   - launch_quarter (1-4)")
print(f"   - days_since_previous_launch (0 to {df['days_since_previous_launch'].max():.0f})")

In [ ]:

print("BLOCK 18: CATEGORICAL ENCODING")


# Encode rocket_type
print("\n Encoding rocket_type...")
rocket_type_map = {'PSLV': 0, 'GSLV': 1, 'LVM3': 2}
df['rocket_type_encoded'] = df['rocket_type'].map(rocket_type_map)
print("   Mapping:", rocket_type_map)
print("   Encoded distribution:")
print(df['rocket_type_encoded'].value_counts().sort_index().to_string())

# Encode launch_site
print("\n Encoding launch_site...")
launch_sites = df['launch_site'].unique()
print(f"   Unique sites: {launch_sites}")
launch_site_map = {site: idx for idx, site in enumerate(sorted(launch_sites)) if pd.notna(site)}
launch_site_map[None] = -1  # For missing values
df['launch_site_encoded'] = df['launch_site'].map(launch_site_map).fillna(-1).astype(int)
print("   Mapping:", {k: v for k, v in launch_site_map.items() if k is not None})

# Encode orbit type
print("\n Encoding orbit...")
# Clean orbit values
df['orbit_clean'] = df['orbit'].fillna('Unknown').str.strip()

# Group similar orbits
orbit_groups = {
    'LEO': ['LEO', 'Low Earth', 'Low Earth Orbit'],
    'GTO': ['GTO', 'Geosynchronous', 'Geosynchronous transfer'],
    'SSO': ['SSO', 'Sun-synchronous'],
    'Polar': ['Polar', 'Polar orbit'],
    'Unknown': ['Unknown', '', 'None']
}

def map_orbit(orbit_val):
    """Map orbit to standardized category."""
    if pd.isna(orbit_val) or orbit_val == 'Unknown':
        return 'Unknown'
    for group, variants in orbit_groups.items():
        if any(variant.lower() in orbit_val.lower() for variant in variants):
            return group
    return 'Other'

df['orbit_category'] = df['orbit_clean'].apply(map_orbit)
orbit_cat_map = {cat: idx for idx, cat in enumerate(sorted(df['orbit_category'].unique()))}
df['orbit_encoded'] = df['orbit_category'].map(orbit_cat_map)

print("   Orbit categories:")
print(df['orbit_category'].value_counts().to_string())

# Encode rocket configuration
print("\n Encoding rocket_configuration...")
config_map = {}
for idx, config in enumerate(sorted(df['rocket_configuration'].dropna().unique())):
    config_map[config] = idx
config_map[None] = -1
df['rocket_config_encoded'] = df['rocket_configuration'].map(config_map).fillna(-1).astype(int)
print(f"   Total configurations: {len(config_map)-1}")

print("\n Categorical encoding complete!")
print(f"   New encoded columns: 4")

In [ ]:
# @title Block 19: Payload Features
print("BLOCK 19: PAYLOAD FEATURES")

# Extract numeric payload mass
print("\n Extracting payload mass...")

def extract_payload_mass(mass_str):
    """Extract numeric mass in kg from various formats."""
    if pd.isna(mass_str):
        return None
    mass_str = str(mass_str).strip()
    # Find all numbers in the string
    numbers = re.findall(r'\d+\.?\d*', mass_str)
    if not numbers:
        return None
    # If multiple masses (multiple payloads), sum them
    masses = [float(n) for n in numbers]
    return sum(masses)

df['payload_mass_kg'] = df['payload_mass'].apply(extract_payload_mass)

# Statistics
valid_masses = df['payload_mass_kg'].dropna()
print(f"   Extracted mass for {len(valid_masses)} / {len(df)} launches")
print(f"   Mass statistics (kg):")
print(f"      Min:     {valid_masses.min():.2f}")
print(f"      Max:     {valid_masses.max():.2f}")
print(f"      Mean:    {valid_masses.mean():.2f}")
print(f"      Median:  {valid_masses.median():.2f}")

# Count number of payloads
print("\nCounting payloads per launch...")

def count_payloads(payload_str):
    """Count number of payloads in launch."""
    if pd.isna(payload_str):
        return 0
    payload_str = str(payload_str)
    # Count newlines (multiple payloads separated by newlines)
    newline_count = payload_str.count('\n') + 1
    # Also check for common separators
    if ',' in payload_str or ';' in payload_str:
        separators = max(payload_str.count(','), payload_str.count(';'))
        return max(newline_count, separators + 1)
    return newline_count

df['payload_count'] = df['payload'].apply(count_payloads)
print("   Payload count distribution:")
print(df['payload_count'].value_counts().sort_index().head(10).to_string())

# Payload complexity score
print("\nCreating payload complexity score...")

# Complexity = (payload_count * 0.3) + (normalized_mass * 0.7)
max_mass = df['payload_mass_kg'].max()
# Guard against division by zero or all-missing payload mass
if pd.isna(max_mass) or max_mass == 0:
    median_mass = df['payload_mass_kg'].median()
    if pd.isna(median_mass) or median_mass == 0:
        max_mass = 1.0
    else:
        max_mass = median_mass
df['payload_mass_normalized'] = (
    df['payload_mass_kg'].fillna(df['payload_mass_kg'].median()) / max_mass
)
df['payload_complexity'] = (
    df['payload_count'] * 0.3
    + df['payload_mass_normalized'] * 0.7
 )

print(
    f"   Complexity score range: ",
    f"{df['payload_complexity'].min():.3f} to {df['payload_complexity'].max():.3f}"
)

# Binary feature: single vs multiple payload
df['is_multi_payload'] = (df['payload_count'] > 1).astype(int)
multi_payload_pct = df['is_multi_payload'].mean() * 100
print(
    f"\n   Multi-payload launches: ",
    f"{df['is_multi_payload'].sum()} / {len(df)} ({multi_payload_pct:.1f}%)"
)

print("\n Payload feature engineering complete!")


In [ ]:
# @title Block 20: Target Variable Creation


print("BLOCK 20: TARGET VARIABLE CREATION")


# Inspect current outcomes
print("\n Current launch_outcome distribution:")
print(df['launch_outcome'].value_counts().to_string())

# Create binary target: Success (1) vs Failure/Partial (0)
print("\n Creating binary target variable (launch_success)...")

def map_to_binary(outcome):
    """Map outcome to binary success (1) or failure (0)."""
    if pd.isna(outcome):
        return None

    outcome_lower = str(outcome).lower().strip()

    if 'success' in outcome_lower and 'partial' not in outcome_lower:
        return 1
    elif 'failure' in outcome_lower or 'partial' in outcome_lower:
        return 0
    elif 'scheduled' in outcome_lower or 'cancelled' in outcome_lower:
        return None  # Exclude future/cancelled launches
    else:
        return None

df['launch_success'] = df['launch_outcome'].apply(map_to_binary)

# Statistics
success_count = (df['launch_success'] == 1).sum()
failure_count = (df['launch_success'] == 0).sum()
unknown_count = df['launch_success'].isna().sum()
total_classified = success_count + failure_count

print(f"Binary classification results:")
if total_classified > 0:
    print(f"  Success:  {success_count}/{total_classified} ({success_count/total_classified*100:.1f}%)")
    print(f"  Failure:  {failure_count}/{total_classified} ({failure_count/total_classified*100:.1f}%)")
else:
    print(f"  Success:  {success_count}/{total_classified} (N/A)")
    print(f"  Failure:  {failure_count}/{total_classified} (N/A)")
print(f"  Unknown:  {unknown_count}")

# Create multi-class target
print("\n Creating multi-class target variable (outcome_category)...")

def map_to_multiclass(outcome):
    """Map outcome to multi-class categories."""
    if pd.isna(outcome):
        return 'Unknown'

    outcome_lower = str(outcome).lower().strip()

    if 'success' in outcome_lower and 'partial' not in outcome_lower:
        return 'Success'
    elif 'partial' in outcome_lower:
        return 'Partial'
    elif 'failure' in outcome_lower:
        return 'Failure'
    else:
        return 'Unknown'

df['outcome_category'] = df['launch_outcome'].apply(map_to_multiclass)

# Encode multi-class
outcome_cat_map = {'Success': 2, 'Partial': 1, 'Failure': 0, 'Unknown': -1}
df['outcome_encoded'] = df['outcome_category'].map(outcome_cat_map)

print(f"Multi-class distribution:")
print(df['outcome_category'].value_counts().to_string())

# Class balance analysis
print("\n CLASS BALANCE ANALYSIS:")
if total_classified > 0:
    imbalance_ratio = success_count / failure_count if failure_count > 0 else float('inf')
    print(f"  Success/Failure ratio: {imbalance_ratio:.2f}:1")

    if imbalance_ratio > 3:
        print("  WARNING: Significant class imbalance detected!")
        print("  Consider using stratified sampling or SMOTE in ML phase")
    else:
        print("  Class balance is acceptable")


In [ ]:
# @title Block 21: Feature Engineering


print("BLOCK 21: ADVANCED FEATURE ENGINEERING")


# Sort by datetime for sequential features
df = df.sort_values('launch_datetime').reset_index(drop=True)

# Experience index (cumulative launches by rocket type)
print("\n Creating experience_index...")
df['experience_index'] = df.groupby('rocket_type').cumcount() + 1

print(f"Experience by rocket type:")
for rocket in df['rocket_type'].unique():
    max_exp = df[df['rocket_type'] == rocket]['experience_index'].max()
    print(f"  - {rocket}: {max_exp} total launches")

# Configuration maturity (launches per configuration)
print("\n Creating configuration_maturity...")
config_counts = df.groupby('rocket_configuration').cumcount() + 1
df['configuration_maturity'] = config_counts

print(f"Configuration maturity range: 1 to {df['configuration_maturity'].max()}")

# Mission frequency (launches per year for each rocket)
print("\n Creating launch_frequency (launches per year)...")
df['launch_frequency'] = df.groupby(['rocket_type', 'launch_year']).cumcount() + 1

# Seasonal features
print("\n Creating seasonal features...")
# Monsoon season in India (June-September)
df['is_monsoon_season'] = df['launch_month'].isin([6, 7, 8, 9]).astype(int)

monsoon_launches = df['is_monsoon_season'].sum()
print(f"Monsoon season launches: {monsoon_launches}/{len(df)} ({monsoon_launches/len(df)*100:.1f}%)")

# Launch window (morning/afternoon/evening/night)
def categorize_launch_time(hour):
    """Categorize launch by time of day."""
    if pd.isna(hour):
        return 'Unknown'
    if 0 <= hour < 6:
        return 'Night'
    elif 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    else:
        return 'Evening'

df['launch_time_category'] = df['launch_hour'].apply(categorize_launch_time)
launch_time_map = {'Morning': 0, 'Afternoon': 1, 'Evening': 2, 'Night': 3, 'Unknown': -1}
df['launch_time_encoded'] = df['launch_time_category'].map(launch_time_map)

print(f"Launch time distribution:")
print(df['launch_time_category'].value_counts().to_string())

# Payload risk score
print("\n Creating payload_risk_score...")
# Risk = (payload_complexity * 0.4) + (is_multi_payload * 0.3) + (normalized_mass * 0.3)
df['payload_risk_score'] = (
    df['payload_complexity'] * 0.4 +
    df['is_multi_payload'] * 0.3 +
    df['payload_mass_normalized'] * 0.3
)

print(f"Risk score range: {df['payload_risk_score'].min():.3f} to {df['payload_risk_score'].max():.3f}")

# Time since first launch (maturity of program)
print("\nCreating program_maturity (years since first launch)...")
first_launch_date = df['launch_datetime'].min()
df['days_since_first_launch'] = (df['launch_datetime'] - first_launch_date).dt.days
df['years_since_first_launch'] = df['days_since_first_launch'] / 365.25

print(f"  Program duration: {df['years_since_first_launch'].max():.1f} years")

print("\nAdvanced feature engineering complete!")
print(f"  New features created: 10")

In [ ]:
# @title Block 22: Missing Value Handling


print("BLOCK 22: MISSING VALUE HANDLING")


# Identify columns with missing values
print("\n Missing value summary:")
missing_summary = df.isnull().sum()
missing_cols = missing_summary[missing_summary > 0].sort_values(ascending=False)

if len(missing_cols) > 0:
    for col in missing_cols.index:
        count = missing_cols[col]
        pct = count / len(df) * 100
        print(f"  {col:35s}: {count:3d} missing ({pct:5.1f}%)")
else:
    print("  No missing values in critical features!")

# Define imputation strategy
print("\n Applying imputation strategies...")

imputation_log = []

# Numerical features: impute with median
numerical_features = [
    'payload_mass_kg', 'payload_complexity', 'payload_mass_normalized',
    'days_since_previous_launch', 'payload_risk_score'
]

for col in numerical_features:
    if col in df.columns and df[col].isna().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        imputation_log.append(f"  {col}: filled with median ({median_val:.2f})")

# Categorical encoded features: impute with -1 (Unknown)
categorical_encoded = [
    'launch_site_encoded', 'orbit_encoded', 'rocket_config_encoded',
    'launch_time_encoded'
]

for col in categorical_encoded:
    if col in df.columns and df[col].isna().any():
        df[col].fillna(-1, inplace=True)
        imputation_log.append(f"  {col}: filled with -1 (Unknown)")

# Temporal features: forward fill for missing datetime-derived features
temporal_features = [
    'launch_year', 'launch_month', 'launch_day_of_week',
    'launch_hour', 'launch_quarter'
]

for col in temporal_features:
    if col in df.columns and df[col].isna().any():
        df[col] = df[col].ffill()  # Modern syntax

        imputation_log.append(f"  {col}: forward filled")

# Print imputation log
if imputation_log:
    print("\n Imputation applied:")
    for log in imputation_log:
        print(log)
else:
    print("\n  No imputation needed!")

# Verify no critical missing values remain
print("\n Verification after imputation:")
critical_features = [
    'rocket_type_encoded', 'launch_success', 'payload_mass_kg',
    'experience_index', 'configuration_maturity'
]

all_clear = True
for col in critical_features:
    if col in df.columns:
        missing = df[col].isna().sum()
        if missing > 0:
            print(f"  {col}: {missing} still missing")
            all_clear = False

if all_clear:
    print("  All critical features have no missing values!")

In [ ]:
# @title Block 23: Data Validation


print("BLOCK 23: DATA VALIDATION")


# Check for data leakage
print("\n CHECKING FOR DATA LEAKAGE...")
print("  Ensuring no future information in features...")

# Verify dates are in the past (handle timezone-aware and naive datetimes)
launch_dt = pd.to_datetime(df['launch_datetime'], errors='coerce')
def _to_utc_naive(ts):
    if pd.isna(ts):
        return ts
    try:
        if getattr(ts, 'tz', None) is None:
            return ts
        return ts.tz_convert('UTC').tz_localize(None)
    except Exception:
        return ts
launch_dt_norm = launch_dt.apply(_to_utc_naive)
now_utc_naive = pd.Timestamp.now(tz='UTC').tz_localize(None)
future_launches = df[launch_dt_norm > now_utc_naive]
if len(future_launches) > 0:
    print(f"  WARNING: {len(future_launches)} future launches detected!")
    print("  These should be excluded from training set")
else:
    print("  No future launches in dataset")

# Check for outliers in numerical features
print("\n OUTLIER DETECTION (using IQR method)...")

numerical_cols = [
    'payload_mass_kg', 'experience_index', 'configuration_maturity',
    'days_since_previous_launch', 'payload_complexity', 'payload_risk_score'
]

outlier_summary = []
for col in numerical_cols:
    if col in df.columns and df[col].notna().any():
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)

        if outlier_count > 0:
            outlier_summary.append({
                'Feature': col,
                'Outliers': outlier_count,
                'Percentage': f"{outlier_count/len(df)*100:.1f}%"
            })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary)
    print(outlier_df.to_string(index=False))
    print("\n  Note: Outliers are kept (they may be valid extreme cases)")
else:
    print("  No significant outliers detected")

# Verify encoded values are in expected ranges
print("\n VALIDATING ENCODED FEATURES...")

encoding_checks = {
    'rocket_type_encoded': (0, 2),
    'outcome_encoded': (-1, 2),
    'launch_time_encoded': (-1, 3),
}

encoding_valid = True
for col, (min_val, max_val) in encoding_checks.items():
    if col in df.columns:
        col_non_na = df[col].dropna()
        if col_non_na.empty:
            print(f"    {col}: no values to validate")
            continue
        actual_min = col_non_na.min()
        actual_max = col_non_na.max()

        if actual_min < min_val or actual_max > max_val:
            print(f"     {col}: range [{actual_min}, {actual_max}] outside expected [{min_val}, {max_val}]")
            encoding_valid = False
        else:
            print(f"    {col}: range [{actual_min:.0f}, {actual_max:.0f}] valid")

# Check data types
print("\n VERIFYING DATA TYPES...")
expected_numeric = numerical_cols + list(encoding_checks.keys())

type_issues = []
for col in expected_numeric:
    if col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            type_issues.append(col)

if type_issues:
    print(f"  Non-numeric columns: {', '.join(type_issues)}")
else:
    print("  All expected columns are numeric")

# Final shape check
print("\n FINAL DATASET SHAPE:")
print(f"  Rows: {df.shape[0]}")
print(f"  Columns: {df.shape[1]}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

print("\n Data validation complete!")

In [ ]:
# @title Block 24: Export Cleaned Data

import json
from datetime import datetime
from pathlib import Path

print("BLOCK 24: EXPORT CLEANED DATA")

# Define output paths (use BASE_DIR from Block 15)
CLEANED_DIR = Path(BASE_DIR) / 'Dataset' / 'processed'
# Create directory (ensure processed folder exists)
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATA_PATH = CLEANED_DIR / 'isro_launch_history_cleaned.csv'
METADATA_PATH = CLEANED_DIR / 'feature_metadata.json'
REPORT_PATH = CLEANED_DIR / 'preprocessing_report.txt'

# Select features for ML (exclude original text columns)
ml_features = [
    # Identifiers
    'flight_number', 'rocket_type',

    # Target variables
    'launch_success', 'outcome_category', 'outcome_encoded',

    # Temporal features
    'launch_datetime', 'launch_year', 'launch_month', 'launch_day_of_week',
    'launch_hour', 'launch_quarter', 'days_since_previous_launch',
    'years_since_first_launch',

    # Categorical encoded
    'rocket_type_encoded', 'launch_site_encoded', 'orbit_encoded',
    'rocket_config_encoded', 'launch_time_encoded',

    # Numerical features
    'payload_mass_kg', 'payload_count', 'payload_complexity',
    'payload_mass_normalized', 'is_multi_payload', 'payload_risk_score',

    # Program maturity
    'experience_index', 'configuration_maturity', 'launch_frequency',

    # Seasonal
    'is_monsoon_season'
]

# Filter to existing columns
ml_features = [col for col in ml_features if col in df.columns]
df_cleaned = df[ml_features].copy()

# Save cleaned data
print("\n Saving cleaned dataset...")
df_cleaned.to_csv(CLEANED_DATA_PATH, index=False)
print(f"    Saved to: {CLEANED_DATA_PATH}")
print(f"   Shape: {df_cleaned.shape[0]} rows × {df_cleaned.shape[1]} columns")

# Create feature metadata
print("\n Creating feature metadata...")
metadata = {
    'created_date': datetime.now().isoformat(),
    'total_features': len(ml_features),
    'feature_descriptions': {
        'launch_success': 'Binary target (1=Success, 0=Failure)',
        'rocket_type_encoded': '0=PSLV, 1=GSLV, 2=LVM3',
        'experience_index': 'Cumulative launches by rocket type',
        'configuration_maturity': 'Cumulative launches per configuration',
        'payload_complexity': 'Composite score of payload count and mass',
        'payload_risk_score': 'Risk assessment based on payload characteristics',
        'is_monsoon_season': '1 if launch in June-September, 0 otherwise',
        'days_since_previous_launch': 'Days since last launch of same rocket type'
    },
    'encoding_maps': {
        'rocket_type': {'PSLV': 0, 'GSLV': 1, 'LVM3': 2},
        'outcome_category': {'Failure': 0, 'Partial': 1, 'Success': 2},
        'launch_time': {'Morning': 0, 'Afternoon': 1, 'Evening': 2, 'Night': 3}
    },
    'statistics': {
        'total_launches': int(df_cleaned.shape[0]),
        'successful_launches': int((df_cleaned['launch_success'] == 1).sum()) if 'launch_success' in df_cleaned.columns else 0,
        'failed_launches': int((df_cleaned['launch_success'] == 0).sum()) if 'launch_success' in df_cleaned.columns else 0,
        'date_range': {
            'start': str(df_cleaned['launch_datetime'].min()) if 'launch_datetime' in df_cleaned.columns else None,
            'end': str(df_cleaned['launch_datetime'].max()) if 'launch_datetime' in df_cleaned.columns else None
        }
    }
}

with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"    Metadata saved to: {METADATA_PATH}")

# Create preprocessing report
print("\n Creating preprocessing report...")
# Guard percentage calculations when total launches = 0
total_launches = metadata['statistics']['total_launches']
if total_launches > 0:
    success_pct_str = f"{metadata['statistics']['successful_launches']/total_launches*100:.1f}%"
    failure_pct_str = f"{metadata['statistics']['failed_launches']/total_launches*100:.1f}%"
else:
    success_pct_str = "N/A"
    failure_pct_str = "N/A"
report_lines = [
    "=" * 80,
    "ISRO LAUNCH DATA - PREPROCESSING REPORT",
    "=" * 80,
    f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"\n{'='*80}",
    "\n1. DATA OVERVIEW",
    f"   Total Launches: {df_cleaned.shape[0]}",
    f"   Features: {df_cleaned.shape[1]}",
    f"   Date Range: {df_cleaned['launch_year'].min():.0f} - {df_cleaned['launch_year'].max():.0f}",
    "\n2. TARGET VARIABLE DISTRIBUTION",
    f"   Success: {metadata['statistics']['successful_launches']} ({success_pct_str})",
    f"   Failure: {metadata['statistics']['failed_launches']} ({failure_pct_str})",
    "\n3. ROCKET TYPE DISTRIBUTION"
]

for rocket_type in sorted(df_cleaned['rocket_type'].unique()):
    count = (df_cleaned['rocket_type'] == rocket_type).sum()
    pct = count / len(df_cleaned) * 100
    report_lines.append(f"   {rocket_type}: {count} ({pct:.1f}%)")

report_lines.extend([
    "\n4. FEATURE ENGINEERING APPLIED",
    "    Temporal features extracted (year, month, hour, etc.)",
    "    Experience index calculated per rocket type",
    "    Payload complexity and risk scores computed",
    "    Seasonal features added (monsoon season)",
    "    Launch time categories encoded",
    "\n5. DATA QUALITY",
    f"   Missing values in target: {df_cleaned['launch_success'].isna().sum() if 'launch_success' in df_cleaned.columns else 0}",
    f"   Missing values in features: {df_cleaned.drop(columns=['launch_success']).isna().sum().sum() if 'launch_success' in df_cleaned.columns else df_cleaned.isna().sum().sum()}",
    f"   Duplicate rows: {df_cleaned.duplicated().sum()}",
    f"\n{'='*80}",
    "READY FOR EDA AND MODELING",
    "=" * 80
])

report_text = '\n'.join(report_lines)
with open(REPORT_PATH, 'w') as f:
    f.write(report_text)
print(f"    Report saved to: {REPORT_PATH}")

print("\n All files exported successfully!")


In [ ]:
# @title Block 25: Preprocessing Summary


print("BLOCK 25: PREPROCESSING SUMMARY")


# Before/After comparison
print("\n BEFORE → AFTER COMPARISON:")
print(f"   Rows:    {df_raw.shape[0]} → {df_cleaned.shape[0]} (unchanged)")
print(f"   Columns: {df_raw.shape[1]} → {df_cleaned.shape[1]} (+{df_cleaned.shape[1] - df_raw.shape[1]} features)")

# Feature categories
print("\n FEATURE CATEGORIES:")
feature_categories = {
    'Temporal': ['launch_year', 'launch_month', 'launch_hour', 'days_since_previous_launch'],
    'Categorical Encoded': ['rocket_type_encoded', 'launch_site_encoded', 'orbit_encoded'],
    'Payload': ['payload_mass_kg', 'payload_count', 'payload_complexity', 'payload_risk_score'],
    'Program Maturity': ['experience_index', 'configuration_maturity', 'launch_frequency'],
    'Target Variables': ['launch_success', 'outcome_encoded']
}

for category, features in feature_categories.items():
    existing = [f for f in features if f in df_cleaned.columns]
    print(f"   {category:20s}: {len(existing)} features")

# Data quality metrics
print("\n DATA QUALITY METRICS:")
total_cells = df_cleaned.shape[0] * df_cleaned.shape[1]
missing_cells = df_cleaned.isna().sum().sum()
completeness = ((total_cells - missing_cells) / total_cells) * 100

print(f"   Data Completeness: {completeness:.2f}%")
print(f"   Duplicate Rows: {df_cleaned.duplicated().sum()}")
print(f"   Unique Launches: {df_cleaned['flight_number'].nunique()}")

# Quick correlation preview (top 5 features with target)
print("\n TOP 5 FEATURE CORRELATIONS WITH launch_success:")
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
correlations = df_cleaned[numeric_cols].corrwith(df_cleaned['launch_success']).abs().sort_values(ascending=False)
top_corr = correlations[correlations.index != 'launch_success'].head(5)

for idx, (feature, corr) in enumerate(top_corr.items(), 1):
    print(f"   {idx}. {feature:35s}: {corr:.3f}")

# Success rate by rocket type
print("\n SUCCESS RATE BY ROCKET TYPE:")
for rocket in sorted(df_cleaned['rocket_type'].unique()):
    rocket_df = df_cleaned[df_cleaned['rocket_type'] == rocket]
    success_rate = (rocket_df['launch_success'] == 1).mean() * 100
    total = len(rocket_df)
    print(f"   {rocket:6s}: {success_rate:5.1f}% ({total} launches)")

print("\n PREPROCESSING COMPLETE!")
print("\n Output Files:")
print(f"   1. {CLEANED_DATA_PATH}")
print(f"   2. {METADATA_PATH}")
print(f"   3. {REPORT_PATH}")
print("\n READY FOR NEXT PHASE:")
print("   → 03_EDA_Analysis.ipynb")
